# Cuaderno de entrenamiento mediante k-fold
## Configuración del cuaderno


In [ ]:
#Instalación de paquetes
!pip install tf_keras tensorflow numpy matplotlib -q
!pip install coral-ordinal

import os

# Forzar uso de Keras 2 para evitar problemas de compatibilidad con STM32Cube.AI
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import shutil
import numpy as np
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers, Model
import coral_ordinal as coral
from PIL import Image
from sklearn.model_selection import LeaveOneOut

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
)

# Verificar que estamos en Keras 2
assert int(keras.__version__.split('.')[0]) == 2, "No se está usando la versión de Keras2"
print("Usando Keras2")

from google.colab import drive
drive.mount('/content/drive')

import sys

SRC_PATH = "/content/drive/MyDrive/TFG/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

In [ ]:
# Macros

DATASET_ORI_PATH = '/content/drive/MyDrive/TFG/dt_ori'
DATASET_AUG_PATH = '/content/drive/MyDrive/TFG/dt_aug'
FOLDS_PATH = '/content/drive/MyDrive/TFG/folds'
RESULTADOS_PATH = f'{FOLDS_PATH}/resultados'

os.makedirs(FOLDS_PATH,   exist_ok=True)
os.makedirs(RESULTADOS_PATH, exist_ok=True)

IMAGE_SIZE = (480, 270)
INPUT_SHAPE = (224, 224, 3)

NUM_FOLDS = 7
NUM_BATCHES = 32
NUM_EPOCH = 50

SEMAFOROS = np.array(['Benidorm', 'Daroca', 'Delicias', 'FrayLuis', 'Martires', 'MCerralbo', 'Pardinas'])
SEMAFOROS_TEST = np.array (['PJesusO'])

LABELS = {'0':'fluido', '1' : 'moderado', '2' : 'denso', '3' : 'saturado'}
NUM_CLASES = len(LABELS)


## Funciones auxiliares

In [ ]:
from preprocesado import ImageCropY
from dataset import build_dataset
from evaluacion import evaluar_modelo

In [ ]:
preprocesado = keras.Sequential([
    keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name='entrada_camara'),
    ImageCropY(0.5, name = 'crop_y'),
    layers.Resizing(INPUT_SHAPE[0], INPUT_SHAPE[1], name = 'resize'),
    layers.Rescaling(1./127.5, offset=-1.0, name = 'normalizacion')
])


## Definición del modelo: MobileNet

In [ ]:
def build_model() :
  #Definición del modelo
  base_model = keras.applications.MobileNet(
      input_shape= INPUT_SHAPE,
      alpha=0.50,
      include_top=False,
      weights='imagenet'
  )
  base_model.trainable = False

  inputs = keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name='input')
  x = preprocesado(inputs)
  x = base_model(x, training=False)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dropout(0.4)(x)
  x = layers.Dense(128, activation='relu')(x)
  x = layers.Dropout(0.3)(x)
  outputs = coral.CoralOrdinal(num_classes=NUM_CLASES, name='output')(x)

  model = Model(inputs, outputs, name='MobileNet_a050_coral')

  # Compilación del modelo
  model.compile(
      optimizer=keras.optimizers.Adam(1e-3),
      loss=coral.OrdinalCrossEntropy (num_classes=NUM_CLASES),
      metrics=[
          coral.MeanAbsoluteErrorLabels(name='mae_labels')
      ]
  )

  return model


## K-FOLD

In [ ]:
custom_objects = {
    'ImageCropY': ImageCropY,
    'CoralOrdinal': coral.CoralOrdinal,
    'OrdinalCrossEntropy': coral.OrdinalCrossEntropy,
    'MeanAbsoluteErrorLabels': coral.MeanAbsoluteErrorLabels
}

In [ ]:
# Entrenamiento de los k-folds
loo = LeaveOneOut()

resultados_test = []

ds_test = build_dataset(SEMAFOROS_TEST, DATASET_ORI_PATH, LABELS, IMAGE_SIZE, NUM_BATCHES, train=False)

for fold_idx, (train_idx, val_idx) in enumerate(loo.split(SEMAFOROS)):
  sem_train = list(SEMAFOROS[train_idx])
  sem_val   = list(SEMAFOROS[val_idx])

  print(f'FOLD {fold_idx + 1}/{NUM_FOLDS}')
  print(f'Train: {sem_train}')
  print(f'Val: {sem_val}')

  # Carga del dataset
  print('Carga de los dataset: ')
  ds_train = build_dataset(sem_train, DATASET_AUG_PATH, LABELS, IMAGE_SIZE, NUM_BATCHES, train=True)
  ds_val = build_dataset(sem_val, DATASET_ORI_PATH, LABELS, IMAGE_SIZE, NUM_BATCHES, train=False)

  # Entrenamiento
  print('Entrenamiento')
  model = build_model()

  callbacks = [

    # Guarda el mejor modelo
    keras.callbacks.ModelCheckpoint(
        filepath        = f'{FOLDS_PATH}/{fold_idx + 1}/best_model.keras',
        monitor         = 'val_mae_labels',
        save_best_only  = True,
        mode            = 'min',
        verbose         = 1
    ),

    # Early stopping
    keras.callbacks.EarlyStopping(
        monitor              = 'val_mae_labels',
        patience             = 12,
        restore_best_weights = True,
        mode                 = 'min',
        verbose              = 1
    ),

    # Reduce LR
    keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_mae_labels',
        factor   = 0.5,
        patience = 6,
        min_lr   = 1e-6,
        mode     = 'min',
        verbose  = 1
    )
  ]

  history = model.fit(
      ds_train,
      epochs=NUM_EPOCH,
      validation_data=ds_val,
      callbacks=callbacks,
      verbose=1
  )

  with open(f'{FOLDS_PATH}/{fold_idx + 1}/history.json', 'w') as f:
      json.dump(
          {k: [float(v) for v in vals] for k, vals in history.history.items()},
          f,
          indent=2
      )

  # Test del modelo
  print('Test')
  model = keras.models.load_model(
      f'{FOLDS_PATH}/{fold_idx + 1}/best_model.keras',
      custom_objects=custom_objects
  )
  
  fold_output = os.path.join(RESULTADOS_PATH, f'fold_{fold_idx + 1}')
  os.makedirs(fold_output, exist_ok=True)
  metricas_fold = evaluar_modelo(model, ds_test, LABELS, fold_output)

  resultados_test.append({
      'fold':    fold_idx + 1,
      'sem_val': sem_val,
      'loss':    round(float(model.evaluate(ds_test, verbose=0)[0]), 6),
      'mae':     metricas_fold['mae'],
      'acc':     metricas_fold['accuracy'],
      'acc_1off':metricas_fold['acc_1off'],
      'qwk':     metricas_fold['qwk'],
      'sesgo':   metricas_fold['sesgo'],
  })

  del model
  keras.backend.clear_session()


# Guardar resultados de test
with open(os.path.join(RESULTADOS_PATH, 'test_kfolds.json'), 'w') as f:
    json.dump(resultados_test, f, indent=2, ensure_ascii=False)